In [ ]:
from collections import Counter, defaultdict
import os
import pickle
import re
import sys
import numpy as np
import pandas as pd

In [ ]:
train_data = pickle.load(open(os.path.join('.', 'train_data_old.pkl'), 'rb'))
elem_comp = pickle.load(open('../data/elements_compounds.pkl', 'rb')) # Updated elements_compunds list

In [ ]:
mol_regex = re.compile(r'(mol|at|wt|weight|mass)\s*\.?\s*%', re.IGNORECASE)
elem_comp['elements'] += ['Composition', 'Compositions', 'composition', 'compositions']
elem_comp_list = elem_comp['elements'] + elem_comp['compounds'] + ['x', 'X', 'y']


def find_mol_wt_in_text(text):
    if re.search(mol_regex, text):
        return 'mol'
    return ''

def find_elem_comp_var(text, unit):
    for elem in elem_comp_list:
        if elem in text:
            return True
    return False

In [ ]:
# find_elem_comp_var('Ko', 'mol')

In [ ]:
def find_num(string):
    #remove tabs or unnecessary spaces
    try:
        string = string.strip()
    except:
        pass
    #e.g. already in int or float form: 12.5 -> 12.5
    try:
        if string[0] == '-':
            return float(string[1:])*(-1)
        else:
            return float(string)
    except:
        pass
    # e.g. 99.1x10-7
    range_regex = re.compile('^\d+\.?\d*\s*x\s*\d+\.?\d*[-|+]?\s*\d+\.?\d*$')
    try:
#         print('alla')
        ranges_ut = range_regex.search(string).group().split('x')
        ranges = [x.strip() for x in ranges_ut]
        #print(f'ranges = {ranges}')
        if '-' in ranges[1]:
            numu = ranges[1].split('-')
#             print(ranges)
#             print(float(numu[0]), float(numu[1]))
            num = float(ranges[0]) * pow(float(numu[0]), (-float(numu[1])))
            formatted_result = "{:.3e}".format(num)
            try:
                # Try to convert using literal_eval
                return float(formatted_result)
            except (ValueError, SyntaxError):
                # If literal_eval fails, return None or handle the error as needed
                return None
        elif '+' in ranges[1]:
            numu = ranges[1].split('+')
            num = float(ranges[0]) * pow(float(numu[0]), (float(numu[1])))
#             print(num)
            return num
        elif ranges[1].startswith('10') and 3<=len(ranges[1])<=4 and ranges[1].isdigit():
            numu0 = 10
            numu1 = ranges[1][2:]
            num = float(ranges[0]) * pow(float(numu0), (float(numu1)))
            return num
        num = float(ranges[0]) * float(ranges[1])
        formatted_result = "{:.3e}".format(num)
        return formatted_result
    except:
        pass
    #e.g. 12.5 - 13.5 -> 12.5
    range_regex = re.compile('^\d+\.?\d*\s*-\s*\d+\.?\d*')
    try:
        ranges = range_regex.search(string).group().split('-')
        num = float(ranges[0])
#         print(num)
        return num
    except:
        pass
#     print('alla')
    #e.g. 12.2 (5.2) -> 12.2
    bracket_regex = re.compile('(\d+\.?\d*)\s*\(\d*.?\d*\)')
    try:
        extracted_value = float(bracket_regex.search(string).group(1))
        return float(extracted_value)
    except:
        pass
    #e.g. 12.3 ± 0.5 -> 12.3
    plusmin_regex = re.compile('^(\d+\.?\d*)(\s*[±+-]+\s*\d+\.?\d*)')
    try:
        extracted_value = float(plusmin_regex.search(string).group(1))
        return extracted_value
    except AttributeError:
        pass
    #e.g. <0.05 -> 0.05  |  >72.0 -> 72.0    | ~12 -> 12
    lessthan_roughly_regex = re.compile('([<]|[~]|[>])=?\s*\d+\.*\d*')
    try:
        extracted_value = lessthan_roughly_regex.search(string).group()
        num_regex = re.compile('\d+\.*\d*')
        extracted_value = num_regex.search(extracted_value).group()
        return float(extracted_value)
    except:
        pass
    # e.g. 0.4:0.6 (ratios)
    if ':' in string:
        split = string.split(":")
        try:
            extracted_value = round(float(split[0])/float(split[1]), 3)
            return extracted_value
        except:
            pass
    # e.g. 220 [29] --> 220.0, where citations given, rejecting ab 220 [29] or 7 220 [29]
    if '[' in string and ']' in string:
        try:
            extracted_value = string[:string.index('[')]
            return float(extracted_value)
        except:
            pass
    # e.g. 723 (first peak or other text) --> 723.0
    if '(' in string and ')' in string:
        try:
            extracted_value = string[:string.index('(')]
            return float(extracted_value)
        except:
            pass
    # e.g. '150K' or '150 degC' --> 150.0 but not '1350degC/2h' or 'njn 150 njn'
    try:
        exact_number_regex = re.compile('^(\d+\.?\d*)\s*[a-zA-Z]+$')
        # Using search to find the first occurrence of the pattern in the string
        match = exact_number_regex.search(string)
#         print('Match')
#         print(match)
        
        # If a match is found, extract the numeric part and convert to float
        if match:
            numeric_part = match.group(1)
            extracted_value = float(numeric_part)
            return extracted_value
    except:
        pass
    return None


def process_2d_list_for_numbers(table):
    return [[find_num(cell) for cell in row] for row in table]

In [ ]:
from regex_lib import *

extracted_regex = pickle.load(open(os.path.join('.', 'extracted_regex.pkl'), 'rb'))
elem_comp['elements'] = elem_comp['elements'][:-4]
all_elements, all_compounds = elem_comp['elements'], elem_comp['compounds']
sorted_compounds_elements = sorted(all_compounds + all_elements, key=len, reverse=True)
comp_pattern = re.compile(r'(?:^|[^a-zA-Z])(' + '|'.join(sorted_compounds_elements) + r')(?:[^a-zA-Z]|$)')
#comp_pattern = re.compile(r'(?<!\w)(' + '|'.join(sorted_compounds_elements) + r')(?!\w)')
comp_pattern_1 = re.compile(r'(' + '|'.join(sorted_compounds_elements) + r')')
var_pattern = re.compile(r'(?:^|[^\w-])(' + '|'.join(comp_vars) + r')')
var_pattern_1 = re.compile(r'(' + '|'.join(comp_vars) + r')')

def add_space_before_mol_percent(text):
    # The pattern looks for mol% preceded by a letter or digit and captures that character
    pattern = re.compile(r'([A-Za-z0-9])(mol%|wt%|at%)')
    
    # This replacement adds a space between the captured letter/digit and "mol%"
    modified_text = re.sub(pattern, r'\1 \2', text)
    
    return modified_text

def get_comp_vars_and_nums(table):
    comps, vars, nums = [], [], []
    for r in table:
        r_comps, r_vars, r_nums = [], [], []
        for cell in r:
            cell = add_space_before_mol_percent(cell)
            found_compounds = re.findall(comp_pattern, cell)
#             print(f'found_compounds = {found_compounds}')
            r_comps.append(found_compounds)
            subs_cell = re.sub(comp_pattern_1, ' ', cell)
            found_vars = list(set(m.group(1) for m in re.finditer(var_pattern, subs_cell)))
            r_vars.append(found_vars)
            subs_cell = re.sub(var_pattern_1, ' ', subs_cell)
            cell_nums = re.findall(num, subs_cell)
            cell_nums = list(map(float, cell_nums))
            r_nums.append(cell_nums[0] if len(cell_nums) > 0 else None)
        comps.append(r_comps)
        vars.append(r_vars)
        nums.append(r_nums)
    return comps, vars, nums


In [ ]:
count_col = 0
change_list = set()
for indd, table in enumerate(train_data):
#     print(indd)
    act_table = table['act_table']
#     display(pd.DataFrame(act_table))
    nr, nc = table['num_rows'], table['num_cols']
    if 'row_label' not in table:
        table['row_label'] = [0] * nr
        count_col += 1
    if 'col_label' not in table:
        table['col_label'] = [0] * nc
    row_label, col_label = table['row_label'], table['col_label']
    table['old_col_label'] = list(table['col_label'])
    table['old_row_label'] = list(table['row_label'])
    comps, vari, nums = get_comp_vars_and_nums(act_table)
#     print('Old nums')
#     #display(pd.DataFrame(nums))
#     print(nums)
    nums = process_2d_list_for_numbers(act_table)
#     print('New nums')
#     #display(pd.DataFrame(nums))
#     print(nums)
#     if indd==10:
#         break
    if 2 not in row_label and 3 not in row_label:
        # col-orientation , by default
        headings = act_table[0]
        check = []
        assert len(headings)==nc
#         display(pd.DataFrame(act_table))
#         display(pd.DataFrame(nums))
        for ind, text in enumerate(headings):
            exclusion = ['error', 'ratio', 'ro']
            is_present = any([word in text.lower() for word in exclusion])
            if is_present:
                continue
            unit = find_mol_wt_in_text(text)
            if unit!= '':
                elem_comp_var = find_elem_comp_var(text, unit)
                if table['col_label'][ind]==0 and elem_comp_var:
                    ## write def median for single_col_row
#                     print(nums)
                    nums_array = np.array(nums, dtype=np.float64)
                    nums_array[np.isnan(nums_array)] = np.nan
#                     display(pd.DataFrame(nums))
#                     display(pd.DataFrame(nums_array))
                    num_int = nums_array[1:, ind]
#                     if indd==2209:
#                         display(pd.DataFrame(nums))
#                         display(pd.DataFrame(nums_array))
#                         display(pd.DataFrame(act_table))
                    median = np.nanmedian(num_int)
                    if median>=0.10:
                        table['col_label'][ind] = 2
                        change_list.add(indd)
                    
    else:
        headings = [rows[0] for rows in act_table]
        assert len(headings)==nr
        for ind, text in enumerate(headings):
            exclusion = ['error', 'ratio', 'ro']
            is_present = any([word in text.lower() for word in exclusion])
            if is_present:
                continue
            unit = find_mol_wt_in_text(text)
            if unit!= '':
                elem_comp_var = find_elem_comp_var(text, unit)
                if table['row_label'][ind]==0 and elem_comp_var:
                    ## write def median for single_col_row
                    nums_array = np.array(nums, dtype=np.float64)
                    nums_array[np.isnan(nums_array)] = np.nan
#                     display(pd.DataFrame(act_table))
#                     print(ind)
#                     display(pd.DataFrame(nums))
#                     display(pd.DataFrame(nums_array))
                    num_int = nums_array[ind, 1:]
                    median = np.nanmedian(num_int)
                    if median>=0.10:
                        table['row_label'][ind] = 2
                        change_list.add(indd)

In [ ]:
# print(count_col)
# print(len(change_list)) #370 row orient, 22 col orient
# print(len(train_data))

In [ ]:
def filter_columns_rows_by_label(data, labels, target_label, orient):
    # Identify indices of columns with the target label
    indices = [i for i, label in enumerate(labels) if label == target_label]

    # Filter the 2D list based on the identified indices
    if orient == 'col':
        filtered_data = [[row[i] for i in indices] for row in data]
    elif orient=='row':
        filtered_data = [data[i] for i in indices]
    
    return filtered_data

def sum_rows(filtered_data, orient):
    if orient == 'row':
        # Sum across each column (sum each column)
        # Transpose the filtered_data using zip(*filtered_data)  to iterate over columns as rows
        return [sum(cell for cell in column if cell is not None) for column in zip(*filtered_data)]
    elif orient == 'col':
        # Sum each row
        return [sum(cell for cell in row if cell is not None) for row in filtered_data]
    else:
        raise ValueError("Invalid orientation: choose 'row' or 'col'")

def find_median_of_specified_columns(nums, orient, label):
    n_r, n_c = len(nums), len(nums[0])
#     print(n_r, n_c)
    nums = [[value if value is not None else -1 for value in row] for row in nums]
#     print(nums)
    medians = []
    
    if orient == 'row':
        #Collect values from each column and calculate their mean
        for col_ind in range(n_c):
            col_values = [row[col_ind] for row in nums if row[col_ind] is not None]
            
            if col_values:
                median = np.median(col_values)
            else:
                median = -1
                
#             print(f'median === {median}')

            medians.append(median)
        
    elif orient == 'col':
        #Collect values from each rows and calculate their mean
        for row in nums:
            row_values = [value for value in row if value is not None]
            
            if row_values:
                median = np.median(row_values)
            else:
                median = -1

            medians.append(median)
    
    
    return medians

In [ ]:
for indd, table in enumerate(train_data):
    if indd in change_list:
#         print(indd)
        act_table = table['act_table']
        table['comp_table'] = True
    #     display(pd.DataFrame(act_table))
        nr, nc = table['num_rows'], table['num_cols']
        nums = process_2d_list_for_numbers(act_table)
        row_label, col_label = table['row_label'], table['col_label']
#         if 2 in col_label and 1 not in row_label:
        if 2 in col_label:
            ##for col orientation
#             print(indd)
#             display(pd.DataFrame(act_table))
#             print(col_label)
#             print(row_label)
#             print(comps)
#             #print(nums)
#             display(pd.DataFrame(nums))
            filtered_data = filter_columns_rows_by_label(nums, col_label, 2, 'col')
#             print(filtered_data)
            sum_fd = sum_rows(filtered_data, 'col')
            sum_fd_non_zero = [x for x in sum_fd if x!=0]
            median_for_ci_pi = np.nanmedian(sum_fd_non_zero)
            if not (0.95<median_for_ci_pi<1.05 or 95<median_for_ci_pi):
                table['sum_less_100'] = 1
            else:
                table['sum_less_100'] = 0
#             print(table['sum_less_100'])
#             print(f'{sum_fd} ==> {median_for_ci_pi}')
#             print('-----------------')
            median_value = find_median_of_specified_columns(filtered_data, 'col', col_label)
#             print(median_value)
            assert len(median_value) == nr
            for ind, median in enumerate(median_value):
                if median>=0 and table['row_label'][ind]==0:
                    table['row_label'][ind] = 1
#             print(row_label)

#         elif 2 in row_label and 1 not in col_label:
        elif 2 in row_label:
            ##for row oritentation
#             display(pd.DataFrame(act_table))
#             print(col_label)
#             print(row_label)
#             print(comps)
#             display(pd.DataFrame(nums))
            filtered_data = filter_columns_rows_by_label(nums, row_label, 2, 'row')
#             print(filtered_data)
            sum_fd = sum_rows(filtered_data, 'row')
            sum_fd_non_zero = [x for x in sum_fd if x!=0]
            median_for_ci_pi = np.nanmedian(sum_fd_non_zero)
            if not (0.95<median_for_ci_pi<1.05 or 95<median_for_ci_pi):
                table['sum_less_100'] = 1
            else:
                table['sum_less_100'] = 0
#             print(table['sum_less_100'])
#             print(f'{sum_fd} ==> {median_for_ci_pi}')
#             print('-----------------')
            median_value = find_median_of_specified_columns(filtered_data, 'row', row_label)
#             print(median_value)
            assert len(median_value) == nc
            for ind, median in enumerate(median_value):
                if median>=0 and table['col_label'][ind] == 0:
                    table['col_label'][ind] = 1
#             print(col_label)
#             break

In [ ]:
count_pi=0
for indd, table in enumerate(train_data):
#     print(table['col_label'])
    if indd in change_list:
        #populate_edges(table)
        if table['sum_less_100']:
            count_pi +=1

In [ ]:
# count = 0
# for indd, table in enumerate(train_data):
#     print(table['col_label'])
#     if indd not in change_list:
#         if not table['sum_less_100'] and table['comp_table'] and not table['regex_table']:
#             if table['num_rows']<7 and table['num_cols']<7 and 2 in table['row_label']:
#                 count+=1
#                 display(pd.DataFrame(table['act_table']))
#                 print(table['act_table'])
#                 print(table['row_label'])
#                 print(table['col_label'])
#                 print(table['edge_list'])
#                 if count==7:
#                     break

In [ ]:
# [(17, 5), (10, 6), (13, 5), (14, 6), (18, 6), (21, 5), (22, 6), (9, 5)]
# [(9, 5), (13, 5), (17, 5), (21, 5), (10, 6), (14, 6), (18, 6), (22, 6)]

In [ ]:
def generate_edge_list(eg_table, row_label, col_label, orient):
    edge_list = []
    num_rows = len(row_label)
    num_cols = len(col_label)

    # Find the fixed destination row for the column orientation case
    if orient == 'col':
        flattened_table = [item for sublist in eg_table for item in sublist]
        dest_row = None
        for i in range(num_rows):
            if row_label[i] == 1:
                if i == 0:
                    dest_row = 0
                else:
                    dest_row = i - 1
                break
        if dest_row is None:
            dest_row = 0  # If no row label is 1, set destination row to 0

        # Iterate through columns first since it's column orientation
        for col in range(num_cols):
            if col_label[col] == 2:
                for row in range(num_rows):
                    if row_label[row] == 1:
                        source = row * num_cols + col
                        destination = dest_row * num_cols + col
                        if find_num(flattened_table[source]) is not None:
                            if find_num(flattened_table[source])>0:
                                edge_list.append((source, destination))
    elif orient == 'row':
#         transpose_table = [list(row) for row in zip(*eg_table)]
        flattened_table = [item for sublist in eg_table for item in sublist]
        #print(flattened_table)
        dest_col = None
        for i in range(num_cols):
            if col_label[i] == 1:
                if i == 0:
                    dest_col = 0
                else:
                    dest_col = i - 1
                break
        if dest_col is None:
            dest_col = 0  # If no column label is 2, set destination column to 0

        # Iterate through rows first since it's row orientation
        for row in range(num_rows):
            if row_label[row] == 2:
                for col in range(num_cols):
                    if col_label[col] == 1:
                        source = row * num_cols + col
                        destination = row * num_cols + dest_col
                        if find_num(flattened_table[source]) is not None:
                            if find_num(flattened_table[source])>0:
                                edge_list.append((source, destination))
    else:
        raise ValueError("Invalid orientation: choose 'row' or 'col'")
    
    return edge_list

In [ ]:
print(len(change_list))
for indd, table in enumerate(train_data):
#     print(table['col_label'])
    if indd in change_list:
        eg_table = table['act_table']
        row_label = table['row_label']
        col_label = table['col_label']
        if 2 in col_label:
            orient = 'col'
        elif 2 in row_label:
            orient = 'row'
        table['edge_list'] = generate_edge_list(eg_table, row_label, col_label, orient)

In [ ]:
pickle.dump(train_data, open('train_data.pkl', 'wb'))